# Dynamic Trust Evaluation

**Dr. Nidhika Chauhan**  
nidhi29.chauhan@gmail.com

In [ ]:
import hashlib, json, time, tracemalloc, platform
from bisect import bisect_left, insort
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as student_t, ttest_rel, wilcoxon
from statsmodels.stats.proportion import proportion_confint
from sklearn.metrics import confusion_matrix, roc_curve, precision_recall_curve, roc_auc_score, average_precision_score, auc
from IPython.display import display
N_NODES, T_INTERVALS, N_RUNS = (500, 1000, 20)
MALICIOUS_RATIO, HISTORY_WINDOW = (0.25, 10)
VM_PER_INTERVAL, CPU_CAPACITY = (200, 100.0)
WARMUP, TRUST_REFRESH_INTERVAL = (50, 50)
LEDGER_DELTA_THRESHOLD, MODELED_CONSENSUS_MS, MODELED_RECORDS_PER_SECOND = (0.02, 10.0, 5000.0)
IDLE_POWER, MAXIMUM_POWER, RERUN_ENERGY_PER_CPU = (100.0, 250.0, 2.0)
CALIBRATION_SEEDS = tuple(range(900, 905))
REPORT_SEEDS = tuple(range(20260, 20260 + N_RUNS))
THRESHOLDS = np.round(np.arange(0.5, 0.951, 0.05), 2)
SENSITIVITIES = (4.0, 4.5, 5.0)
BEHAVIORS = ('on_off', 'always', 'gradual', 'random', 'recovered', 'collusion', 'sybil')
STRATEGIES = ('gbf', 'static_trust', 'proposed', 'periodic_trust', 'no_smoothing')
SCENARIOS = {'normal': VM_PER_INTERVAL, 'capacity_stress': 4500}
OUTPUT_DIR = Path.cwd() / 'evaluation_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.precision', 4)
print('Execution platform:', platform.platform(), platform.python_version())

## AHP weights and their provenance

In [ ]:
TRUST_PARAMETERS = ['CPU', 'Memory', 'Network', 'Migration', 'SLA', 'Abnormality']
AHP_MATRIX = np.array([
    [1, 2, 2, 2, 1, 2],
    [0.5, 1, 1, 1, 0.5, 1],
    [0.5, 1, 1, 1, 0.5, 1],
    [0.5, 1, 1, 1, 0.5, 1],
    [1, 2, 2, 2, 1, 2],
    [0.5, 1, 1, 1, 0.5, 1]
])

def ahp_weights(matrix):
    weights = (matrix / matrix.sum(axis=0)).mean(axis=1)
    principal_eigenvalue = np.mean(matrix @ weights / weights)
    return (weights, max(0.0, (principal_eigenvalue - len(weights)) / (len(weights) - 1) / 1.24))
AHP_WEIGHTS, AHP_CONSISTENCY_RATIO = ahp_weights(AHP_MATRIX)
assert AHP_CONSISTENCY_RATIO < 0.1
print(f'Author-defined AHP consistency ratio: {AHP_CONSISTENCY_RATIO:.6f}')
display(pd.DataFrame({'parameter': TRUST_PARAMETERS, 'weight': AHP_WEIGHTS}))

## Nodes, behavior and noisy monitoring

In [ ]:
@dataclass
class CloudNode:
    id: int
    is_malicious: bool
    behavior: str = 'honest'
    offset: int = 0
    physical_host: int = 0
    drift: float = 0.0

def initialize_nodes(n_nodes, malicious_ratio, rng):
    malicious_ids = rng.permutation(n_nodes)[:round(n_nodes * malicious_ratio)]
    nodes = [CloudNode(i, False, physical_host=i) for i in range(n_nodes)]
    for index, node_id in enumerate(malicious_ids):
        nodes[node_id].is_malicious = True
        nodes[node_id].behavior = BEHAVIORS[index % len(BEHAVIORS)]
        nodes[node_id].offset = int(rng.integers(0, 50))
    sybils = [n for n in nodes if n.behavior == 'sybil']
    for index, node in enumerate(sybils):
        node.physical_host = sybils[index // 3 * 3].id
    return nodes

def monitor_resources(nodes, interval, rng, base, phase):
    node_count = len(nodes)
    hosts = np.array([node.physical_host for node in nodes])
    actual = np.clip(
        base + 12 * np.sin(2 * np.pi * interval / T_INTERVALS + phase)[:, None] + rng.normal(0, 3, (node_count, 3)) + (rng.random((node_count, 1)) < 0.06) * rng.uniform(15, 35, (node_count, 1)),
        1,
        100
    )
    actual = actual[hosts]
    attacking, magnitudes = ([], [])
    for node in nodes:
        behavior = node.behavior
        node.drift = min(1.0, max(0.0, (interval - 50) / 300)) if behavior == 'gradual' else 0.0
        active = (interval + node.offset) % 50 > 30 if behavior == 'on_off' else interval > 50 if behavior == 'gradual' else rng.random() < 0.35 if behavior == 'random' else 150 <= interval <= 450 if behavior == 'recovered' else behavior in ('always', 'collusion', 'sybil')
        attacking.append(active)
        magnitudes.append(node.drift if behavior == 'gradual' else 1.0)
    attacking = np.asarray(attacking, dtype=bool)
    error = rng.normal(0, 1.5, (node_count, 3)) + (rng.random((node_count, 1)) < 0.05) * rng.normal(0, 8, (node_count, 3))
    reported = np.clip(actual + error, 0, 100)
    reporting_bias = rng.uniform([20, 20, 10], [45, 45, 30], (node_count, 3)) * np.array(magnitudes)[:, None]
    reported[attacking] = np.clip(actual[attacking] - reporting_bias[attacking], 0, 100)
    migration_ok = rng.random(node_count) > np.where(attacking, 0.35, 0.05)
    sla_ok = rng.random(node_count) > np.where(attacking, 0.4, 0.06)
    colluding = np.array([node.behavior == 'collusion' for node in nodes])
    migration_ok[colluding] = True
    sla_ok[colluding] = True
    return {
        node.id: dict(
            actual_cpu=float(actual[index, 0]),
            actual_mem=float(actual[index, 1]),
            actual_bw=float(actual[index, 2]),
            reported_cpu=float(reported[index, 0]),
            reported_mem=float(reported[index, 1]),
            reported_bw=float(reported[index, 2]),
            migration_ok=bool(migration_ok[index]),
            sla_ok=bool(sla_ok[index]),
            is_attacking=bool(attacking[index]),
            advertised_capacity=CPU_CAPACITY
        ) for index,
        node in enumerate(nodes)
    }

def generate_vm_requests(interval, rng, base_rate):
    rate = base_rate * (1.0 - 0.5 * np.cos(2 * np.pi * interval / T_INTERVALS))
    return np.clip(rng.lognormal(np.log(8), 0.6, max(1, rng.poisson(rate))), 5, 20)

def build_environment(seed, n_nodes=N_NODES, t_intervals=T_INTERVALS):
    rng = np.random.default_rng(seed)
    nodes = initialize_nodes(n_nodes, MALICIOUS_RATIO, rng)
    base = rng.beta(2, 5, (n_nodes, 3)) * 80 + 5
    phase = rng.uniform(0, 2 * np.pi, n_nodes)
    actual, reported, flags, migration, sla, raw_bytes = ([], [], [], [], [], [])
    for interval in range(t_intervals):
        telemetry = monitor_resources(nodes, interval, rng, base, phase)
        actual.append([[
            measurement['actual_cpu'],
            measurement['actual_mem'],
            measurement['actual_bw']
        ] for measurement in telemetry.values()])
        reported.append([[
            measurement['reported_cpu'],
            measurement['reported_mem'],
            measurement['reported_bw']
        ] for measurement in telemetry.values()])
        flags.append([measurement['is_attacking'] for measurement in telemetry.values()])
        migration.append([measurement['migration_ok'] for measurement in telemetry.values()])
        sla.append([measurement['sla_ok'] for measurement in telemetry.values()])
        raw_bytes.append(len(json.dumps(telemetry, sort_keys=True).encode('utf-8')))
    return dict(
        nodes=nodes,
        actual=np.array(actual),
        reported=np.array(reported),
        y=np.array(flags),
        migration=np.array(migration),
        sla=np.array(sla),
        raw_bytes=np.array(raw_bytes),
        seed=seed,
        T=t_intervals
    )

## Trust computation and attack/recovery timing

In [ ]:
def rolling_mean(values, window=HISTORY_WINDOW, initial=0.0):
    padded = np.concatenate([np.full((window,) + values.shape[1:], initial), values], axis=0)
    cumulative_sum = np.cumsum(padded, axis=0)
    return (cumulative_sum[window:] - cumulative_sum[:-window]) / window

def analyze_behavior(environment, sensitivity):
    discrepancy = np.abs(environment['actual'] - environment['reported']) / 100
    mean_discrepancy = rolling_mean(discrepancy)
    resource = np.clip(1 - sensitivity * mean_discrepancy, 0, 1)
    migration = rolling_mean(environment['migration'].astype(float), initial=1)
    sla = rolling_mean(environment['sla'].astype(float), initial=1)
    abnormality = np.clip(1 - sensitivity * mean_discrepancy.mean(axis=2), 0, 1)
    return np.concatenate(
        [resource, migration[:, :, None], sla[:, :, None], abnormality[:, :, None]],
        axis=2
    )

def compute_trust_scores(features, weights=AHP_WEIGHTS, smoothing=True):
    raw = np.clip(features @ weights, 0, 1)
    if not smoothing:
        return raw
    trust, previous_trust = (np.empty_like(raw), np.ones(raw.shape[1]))
    for interval, row in enumerate(raw):
        smoothing_factor = np.where(row < previous_trust, 0.45, 0.08)
        previous_trust = (1 - smoothing_factor) * previous_trust + smoothing_factor * row
        trust[interval] = previous_trust
    return trust

def detection_counts(attack_labels, flags):
    tn, fp, fn, tp = confusion_matrix(
        attack_labels.ravel(),
        flags.ravel(),
        labels=[False, True]
    ).ravel()
    safe_divide = lambda a, b: float(a / b) if b else np.nan
    return dict(
        tp=int(tp),
        fp=int(fp),
        fn=int(fn),
        tn=int(tn),
        precision=safe_divide(tp, tp + fp),
        recall=safe_divide(tp, tp + fn),
        f1=safe_divide(2 * tp, 2 * tp + fp + fn),
        fpr=safe_divide(fp, fp + tn),
        fnr=safe_divide(fn, fn + tp)
    )

def episode_latencies(attack_labels, flags, nodes):
    records = []
    for node_id, node in enumerate(nodes):
        state = attack_labels[:, node_id]
        starts = np.flatnonzero(state & ~np.r_[False, state[:-1]])
        recoveries = np.flatnonzero(~state & np.r_[False, state[:-1]])
        for kind, edges, target in [('TTD', starts, True), ('TTR', recoveries, False)]:
            for start in edges:
                changes = np.flatnonzero(state[start:] != state[start])
                end = start + changes[0] if len(changes) else len(state)
                matches = np.flatnonzero(flags[start:end, node_id] == target)
                latency = int(matches[0]) if len(matches) else np.nan
                records.append(dict(
                    node=node_id,
                    behavior=node.behavior,
                    kind=kind,
                    start=int(start),
                    latency=latency,
                    censored=not len(matches),
                    already_in_target=bool(flags[start, node_id] == target)
                ))
    return pd.DataFrame(records)

def calibrate(seeds=CALIBRATION_SEEDS, n_nodes=N_NODES, t_intervals=T_INTERVALS):
    rows = []
    for seed in seeds:
        simulation_environment = build_environment(seed, n_nodes, t_intervals)
        for sensitivity in SENSITIVITIES:
            trust = compute_trust_scores(analyze_behavior(simulation_environment, sensitivity))
            for threshold in THRESHOLDS:
                rows.append(dict(
                    seed=seed,
                    sensitivity=sensitivity,
                    threshold=threshold,
                    **detection_counts(simulation_environment['y'], trust < threshold)
                ))
    table = pd.DataFrame(rows)
    ranking = table.groupby(['sensitivity', 'threshold'], as_index=False)[['f1', 'fpr']].mean()
    best = ranking.sort_values(['f1', 'fpr', 'threshold'], ascending=[False, True, True]).iloc[0]
    return (float(best.sensitivity), float(best.threshold), table)

## Simulated hash-chained ledger

In [ ]:
class Block:

    def __init__(self, index, timestamp, records, events, previous_hash):
        self.index, self.timestamp = (index, timestamp)
        self.records, self.events, self.previous_hash = (records, events, previous_hash)
        self.hash = self.compute_hash()

    def payload(self):
        return dict(
            index=self.index,
            timestamp=self.timestamp,
            records=self.records,
            events=self.events,
            previous_hash=self.previous_hash
        )

    def compute_hash(self):
        return hashlib.sha256(json.dumps(self.payload(), sort_keys=True).encode()).hexdigest()

    def size_bytes(self):
        return len(json.dumps(
            dict(**self.payload(), hash=self.hash),
            sort_keys=True
        ).encode('utf-8'))

class TrustBlockchain:

    def __init__(self, delta=LEDGER_DELTA_THRESHOLD, full=False):
        self.chain = [Block(0, -1, {}, [], '0' * 64)]
        self.previous = {}
        self.flags = {}
        self.delta, self.full = (delta, full)

    def add_block(self, interval, scores, threshold):
        score_records, events = ({}, [])
        for node_id, score in enumerate(scores):
            flagged = bool(score < threshold)
            transition = node_id in self.flags and flagged != self.flags[node_id]
            if self.full or node_id not in self.previous or abs(score - self.previous[node_id]) >= self.delta or transition:
                score_records[str(node_id)] = round(float(score), 3)
                self.previous[node_id] = float(score)
            if transition or (node_id not in self.flags and flagged):
                events.append(dict(node=node_id, event='flag' if flagged else 'clear'))
            self.flags[node_id] = flagged
        if not score_records and (not events):
            return None
        block = Block(len(self.chain), int(interval), score_records, events, self.chain[-1].hash)
        self.chain.append(block)
        return block

    def is_valid(self):
        return self.chain[0].previous_hash == '0' * 64 and all((current_block.hash == current_block.compute_hash() for current_block in self.chain)) and all((
            current_block.previous_hash == previous_block.hash for previous_block,
            current_block in zip(self.chain, self.chain[1:])
        ))

def ledger_comparison(trust, threshold, raw_bytes, delta=LEDGER_DELTA_THRESHOLD):
    rows = []
    ledgers = {}
    for name, full in [('delta_scores', False), ('full_scores', True)]:
        ledger = TrustBlockchain(delta, full)
        sizes = []
        records = 0
        for interval, row in enumerate(trust):
            block = ledger.add_block(interval, row, threshold)
            sizes.append(block.size_bytes() if block else 0)
            records += len(block.records) if block else 0
        assert ledger.is_valid()
        rows.append(dict(
            design=name,
            bytes_per_interval=np.mean(sizes),
            total_bytes=sum((block.size_bytes() for block in ledger.chain)),
            blocks=len(ledger.chain) - 1,
            modeled_consensus_ms=(len(ledger.chain) - 1) * MODELED_CONSENSUS_MS,
            modeled_record_service_seconds=records / MODELED_RECORDS_PER_SECOND
        ))
        ledgers[name] = ledger
    rows.append(dict(
        design='raw_telemetry_json',
        bytes_per_interval=np.mean(raw_bytes),
        total_bytes=int(sum(raw_bytes)),
        blocks=np.nan,
        modeled_consensus_ms=np.nan,
        modeled_record_service_seconds=np.nan
    ))
    return (rows, ledgers['delta_scores'])

## Scheduling, physical capacity, energy and overhead

In [ ]:
def greedy_best_fit_allocate(free, requests, trust=None, threshold=0):
    allowed = np.ones(len(free), bool) if trust is None else trust >= threshold
    available = sorted(((float(free[node_id]), node_id) for node_id in range(len(free)) if allowed[node_id]))
    allocations = []
    for demand in requests:
        position = bisect_left(available, (float(demand), -1))
        if position == len(available):
            allocations.append(None)
        else:
            capacity, node_id = available.pop(position)
            insort(available, (capacity - float(demand), node_id))
            allocations.append(node_id)
    return allocations

def power_model(utilization):
    return IDLE_POWER + (MAXIMUM_POWER - IDLE_POWER) * np.clip(utilization, 0, 1) ** 2

def run_simulation(
    strategy,
    environment,
    outcome_seed,
    trust,
    threshold,
    base_rate,
    trust_seconds=0.0,
    trace_memory=True,
    trust_cpu_seconds=0.0,
    trust_peak_bytes=0
):
    node_count, interval_count = (len(environment['nodes']), environment['T'])
    hosts = np.array([x.physical_host for x in environment['nodes']])
    physical = np.unique(hosts)
    rng_requests = np.random.default_rng(environment['seed'] + 100000)
    rng_outcomes = np.random.default_rng(outcome_seed)
    log = []
    start_cpu = time.process_time()
    if trace_memory:
        tracemalloc.start()
    for interval in range(interval_count):
        if strategy == 'gbf':
            active = None
        elif strategy == 'static_trust':
            active = trust[min(interval, WARMUP)]
        elif strategy == 'periodic_trust':
            active = trust[interval // TRUST_REFRESH_INTERVAL * TRUST_REFRESH_INTERVAL]
        else:
            active = trust[interval]
        requests = generate_vm_requests(interval, rng_requests, base_rate)
        uniforms = rng_outcomes.random(len(requests))
        free = CPU_CAPACITY - environment['reported'][interval, :, 0]
        schedule_start = time.perf_counter()
        allocations = greedy_best_fit_allocate(free, requests, active, threshold)
        schedule_time = time.perf_counter() - schedule_start
        actual_load = environment['actual'][interval, :, 0].copy()
        assigned = failed = 0
        rerun = 0.0
        for request_index, node_id in enumerate(allocations):
            if node_id is None:
                continue
            assigned += 1
            host = hosts[node_id]
            actual_load[host] += requests[request_index]
            violation = actual_load[host] > CPU_CAPACITY or uniforms[request_index] < (0.35 if environment['y'][interval, node_id] else 0.03)
            failed += int(violation)
            if violation:
                rerun += requests[request_index] * RERUN_ENERGY_PER_CPU
        log.append(dict(
            t=interval,
            strategy=strategy,
            n_requests=len(requests),
            assigned=assigned,
            throughput=assigned - failed,
            dropped_vms=len(requests) - assigned,
            sla_violations=failed,
            resource_utilization=np.clip(actual_load[physical] / CPU_CAPACITY, 0, 1).mean(),
            energy=power_model(actual_load[physical] / CPU_CAPACITY).sum() + rerun,
            scheduling_time=schedule_time,
            decision_latency_ms=1000 * schedule_time / len(requests)
        ))
    cpu_seconds = time.process_time() - start_cpu
    peak = tracemalloc.get_traced_memory()[1] if trace_memory else np.nan
    if trace_memory:
        tracemalloc.stop()
    interval_results = pd.DataFrame(log)
    counts = detection_counts(
        environment['y'],
        np.zeros_like(environment['y']) if strategy == 'gbf' else np.array([trust[min(interval, WARMUP)] if strategy == 'static_trust' else trust[interval // TRUST_REFRESH_INTERVAL * TRUST_REFRESH_INTERVAL] if strategy == 'periodic_trust' else trust[interval] for interval in range(interval_count)]) < threshold
    )
    totals = {k: float(interval_results[k].mean()) for k in [
        'throughput',
        'dropped_vms',
        'sla_violations',
        'energy',
        'resource_utilization'
    ]}
    totals.update(counts)
    totals.update(
        strategy=strategy,
        decision_latency_ms=1000 * interval_results.scheduling_time.sum() / interval_results.n_requests.sum(),
        dts_time_per_node_us=1000000.0 * trust_seconds / (interval_count * node_count) if strategy != 'gbf' else 0.0,
        cpu_seconds=cpu_seconds + (trust_cpu_seconds if strategy != 'gbf' else 0),
        peak_python_bytes=max(peak, trust_peak_bytes) if trace_memory else np.nan
    )
    return (interval_results, totals)

def measure_trust(environment, smoothing=True):
    tracemalloc.start()
    wall, cpu = (time.perf_counter(), time.process_time())
    features = analyze_behavior(environment, SENSITIVITY)
    scores = compute_trust_scores(features, smoothing=smoothing)
    seconds, cpu_seconds = (time.perf_counter() - wall, time.process_time() - cpu)
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return (features, scores, seconds, cpu_seconds, peak)

## Calibrate once, report on disjoint seeds

In [ ]:
SENSITIVITY, TRUST_THRESHOLD, calibration = calibrate()
assert set(CALIBRATION_SEEDS).isdisjoint(REPORT_SEEDS)
print(f'Calibration seeds {CALIBRATION_SEEDS}: sensitivity={SENSITIVITY}, tau={TRUST_THRESHOLD}')
calibration.to_csv(OUTPUT_DIR / 'calibration.csv', index=False)

In [ ]:
run_records, detection_records, behavior_records, latency_records = ([], [], [], [])
storage_records, ablation_records, curve_records, sweep_records = ([], [], [], [])
random_weights = np.random.default_rng(4242).dirichlet(np.ones(6))
reference_ledger = None
for run, seed in enumerate(REPORT_SEEDS):
    environment = build_environment(seed)
    features, trust, trust_seconds, trust_cpu, trust_peak = measure_trust(environment)
    _, raw_trust, raw_seconds, raw_cpu, raw_peak = measure_trust(environment, smoothing=False)
    warm_environment = dict(environment)
    for key in ['actual', 'reported', 'migration', 'sla']:
        warm_environment[key] = environment[key][:WARMUP + 1]
    _, _, static_seconds, static_cpu, static_peak = measure_trust(warm_environment)
    np.savez_compressed(OUTPUT_DIR / f'trust_run_{run:02d}.npz', trust=trust, y=environment['y'])
    detection_records.append(dict(
        run=run,
        **detection_counts(environment['y'], trust < TRUST_THRESHOLD)
    ))
    honest = np.array([not node.is_malicious for node in environment['nodes']])
    honest_counts = detection_counts(
        environment['y'][:, honest],
        trust[:, honest] < TRUST_THRESHOLD
    )
    detection_records[-1].update(
        honest_fp=honest_counts['fp'],
        honest_tn=honest_counts['tn'],
        honest_fpr=honest_counts['fpr']
    )
    lower_bound, upper_bound = proportion_confint(
        honest_counts['fp'],
        honest_counts['fp'] + honest_counts['tn'],
        method='beta'
    )
    detection_records[-1].update(honest_cp95_low=lower_bound, honest_cp95_high=upper_bound)
    for behavior in BEHAVIORS:
        mask = np.array([node.behavior == behavior for node in environment['nodes']])
        behavior_records.append(dict(
            run=run,
            behavior=behavior,
            **detection_counts(environment['y'][:, mask], trust[:, mask] < TRUST_THRESHOLD)
        ))
    latency = episode_latencies(environment['y'], trust < TRUST_THRESHOLD, environment['nodes'])
    latency['run'] = run
    latency_records.append(latency)
    storage, reference_ledger = ledger_comparison(trust, TRUST_THRESHOLD, environment['raw_bytes'])
    storage_records.extend((dict(run=run, **row) for row in storage))
    for name, weights in [
        ('ahp', AHP_WEIGHTS),
        ('equal', np.ones(6) / 6),
        ('random', random_weights)
    ]:
        scores = compute_trust_scores(features, weights)
        ablation_records.append(dict(
            run=run,
            ablation=name,
            **detection_counts(environment['y'], scores < TRUST_THRESHOLD)
        ))
    ablation_records.append(dict(
        run=run,
        ablation='no_smoothing',
        **detection_counts(environment['y'], raw_trust < TRUST_THRESHOLD)
    ))
    for scenario, rate in SCENARIOS.items():
        for strategy in STRATEGIES:
            seconds, cpu, peak = (0, 0, 0) if strategy == 'gbf' else (static_seconds, static_cpu, static_peak) if strategy == 'static_trust' else (raw_seconds, raw_cpu, raw_peak) if strategy == 'no_smoothing' else (trust_seconds, trust_cpu, trust_peak)
            _, record = run_simulation(
                strategy,
                environment,
                77000 + run,
                raw_trust if strategy == 'no_smoothing' else trust,
                TRUST_THRESHOLD,
                rate,
                seconds,
                trust_cpu_seconds=cpu,
                trust_peak_bytes=peak
            )
            run_records.append(dict(run=run, scenario=scenario, **record))
    attack_labels, score = (environment['y'].ravel(), -trust.ravel())
    fpr, tpr, _ = roc_curve(attack_labels, score)
    precision, recall, _ = precision_recall_curve(attack_labels, score)
    curve_records.append(dict(
        run=run,
        roc_auc=roc_auc_score(attack_labels, score),
        pr_auc=auc(recall, precision),
        average_precision=average_precision_score(attack_labels, score)
    ))
    if run == 0:
        curve_plot = (fpr, tpr, precision, recall)
    for threshold in THRESHOLDS:
        metrics = detection_counts(environment['y'], trust < threshold)
        for scenario, rate in SCENARIOS.items():
            _, scheduled = run_simulation(
                'proposed',
                environment,
                77000 + run,
                trust,
                threshold,
                rate,
                trace_memory=False
            )
            sweep_records.append(dict(
                run=run,
                scenario=scenario,
                threshold=threshold,
                **metrics,
                throughput=scheduled['throughput'],
                dropped_vms=scheduled['dropped_vms'],
                sla_violations=scheduled['sla_violations']
            ))
    print(f'Completed paired reporting run {run + 1}/{N_RUNS}', flush=True)
results = pd.DataFrame(run_records)
detection = pd.DataFrame(detection_records)
behavior_results = pd.DataFrame(behavior_records)
latencies = pd.concat(latency_records, ignore_index=True)
storage_results = pd.DataFrame(storage_records)
ablations = pd.DataFrame(ablation_records)
curves = pd.DataFrame(curve_records)
threshold_sweep = pd.DataFrame(sweep_records)

## Per-run statistics and paired comparisons

In [ ]:
def mean_ci(values):
    finite_values = np.asarray(values, dtype=float)
    finite_values = finite_values[np.isfinite(finite_values)]
    sample_size = len(finite_values)
    mean = float(np.mean(finite_values)) if sample_size else np.nan
    sd = float(np.std(finite_values, ddof=1)) if sample_size > 1 else np.nan
    margin = student_t.ppf(0.975, sample_size - 1) * sd / np.sqrt(sample_size) if sample_size > 1 else np.nan
    return dict(n=sample_size, mean=mean, sd=sd, ci95_low=mean - margin, ci95_high=mean + margin)

def summarize(frame, groups, metrics):
    rows = []
    for keys, group_frame in frame.groupby(groups):
        if not isinstance(keys, tuple):
            keys = (keys,)
        for metric in metrics:
            rows.append(dict(zip(groups, keys), metric=metric, **mean_ci(group_frame[metric])))
    return pd.DataFrame(rows)
metrics = [
    'throughput',
    'dropped_vms',
    'sla_violations',
    'resource_utilization',
    'energy',
    'decision_latency_ms',
    'dts_time_per_node_us',
    'cpu_seconds',
    'peak_python_bytes'
]
summary = summarize(results, ['scenario', 'strategy'], metrics)
detection_summary = pd.DataFrame([dict(
    metric=metric_name,
    **mean_ci(detection[metric_name])
) for metric_name in ['precision', 'recall', 'f1', 'fpr', 'fnr', 'honest_fpr']])
display(summary)
display(detection_summary)
counts = detection[['tp', 'fp', 'fn', 'tn']].sum().astype(int)
confusion = pd.DataFrame(
    [[counts.tn, counts.fp], [counts.fn, counts.tp]],
    index=['Actual inactive', 'Actual attacking'],
    columns=['Predicted clear', 'Predicted flagged']
)
print(f'Confusion matrix: {N_RUNS} runs x {T_INTERVALS} intervals x {N_NODES} nodes = {N_RUNS * T_INTERVALS * N_NODES:,} node-intervals')
display(confusion)
honest_fp = int(detection.honest_fp.sum())
honest_total = int((detection.honest_fp + detection.honest_tn).sum())
honest_fpr_interval = proportion_confint(honest_fp, honest_total, alpha=0.05, method='beta')
honest_fpr_report = pd.DataFrame([dict(
    denominator='honest nodes only',
    fp=honest_fp,
    node_intervals=honest_total,
    pooled_fpr=honest_fp / honest_total,
    cp95_low=honest_fpr_interval[0],
    cp95_high=honest_fpr_interval[1],
    **mean_ci(detection.honest_fpr)
)])
display(honest_fpr_report)
behavior_summary = summarize(behavior_results, ['behavior'], ['recall'])
latency_runs = latencies.groupby(['run', 'behavior', 'kind'], as_index=False).agg(
    latency=('latency', 'mean'),
    censored_rate=('censored', 'mean'),
    already_in_target_rate=('already_in_target', 'mean')
)
latency_summary = summarize(
    latency_runs,
    ['behavior', 'kind'],
    ['latency', 'censored_rate', 'already_in_target_rate']
)
storage_summary = summarize(storage_results, ['design'], ['bytes_per_interval', 'total_bytes'])
ablation_summary = summarize(ablations, ['ablation'], ['precision', 'recall', 'f1', 'fpr'])
display(behavior_summary)
display(latency_summary)
display(storage_summary)
display(ablation_summary)
tests = []
for scenario in SCENARIOS:
    scenario_results = results[results.scenario == scenario]
    for baseline in ['gbf', 'static_trust']:
        for metric in metrics:
            paired = scenario_results[scenario_results.strategy.isin(['proposed', baseline])].pivot(
                index='run',
                columns='strategy',
                values=metric
            ).dropna()
            proposed_values, baseline_values = (
                paired.proposed.to_numpy(),
                paired[baseline].to_numpy()
            )
            if np.all(proposed_values == baseline_values):
                paired_t_pvalue, wilcoxon_pvalue = (1.0, 1.0)
            else:
                paired_t_pvalue = float(ttest_rel(proposed_values, baseline_values).pvalue)
                wilcoxon_pvalue = float(wilcoxon(proposed_values, baseline_values).pvalue)
            tests.append(dict(
                scenario=scenario,
                baseline=baseline,
                metric=metric,
                n=len(proposed_values),
                mean_paired_difference=np.mean(proposed_values - baseline_values),
                ttest_rel_p=paired_t_pvalue,
                wilcoxon_p=wilcoxon_pvalue
            ))
paired_tests = pd.DataFrame(tests)
display(paired_tests)
display(pd.DataFrame([dict(
    metric=metric_name,
    **mean_ci(curves[metric_name])
) for metric_name in ['roc_auc', 'pr_auc', 'average_precision']]))

## Storage sensitivity and figures

In [ ]:
storage_grid_rows = []
for n_nodes in (100, 250, 500):
    for run, seed in enumerate(REPORT_SEEDS):
        if n_nodes == N_NODES:
            saved = np.load(OUTPUT_DIR / f'trust_run_{run:02d}.npz')
            scores = saved['trust']
            raw = np.full(
                T_INTERVALS,
                storage_results.query("run == @run and design == 'raw_telemetry_json'").bytes_per_interval.iloc[0]
            )
        else:
            simulation_environment = build_environment(seed, n_nodes)
            scores = compute_trust_scores(analyze_behavior(simulation_environment, SENSITIVITY))
            raw = simulation_environment['raw_bytes']
        for delta in (0.0, 0.01, 0.02, 0.05, 0.1):
            rows, _ = ledger_comparison(scores, TRUST_THRESHOLD, raw, delta)
            storage_grid_rows.extend((dict(
                run=run,
                N=n_nodes,
                delta=delta,
                **storage_record
            ) for storage_record in rows))
    print(f'Storage grid complete for N={n_nodes}', flush=True)
storage_grid = pd.DataFrame(storage_grid_rows)
storage_grid_summary = summarize(
    storage_grid,
    ['N', 'delta', 'design'],
    ['bytes_per_interval', 'total_bytes']
)
display(storage_grid_summary)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(confusion, cmap='Blues')
for row_index in range(2):
    for column_index in range(2):
        axes[0].text(
            column_index,
            row_index,
            f'{confusion.iloc[row_index, column_index]:,}',
            ha='center'
        )
axes[0].set(
    xticks=[0, 1],
    xticklabels=['Clear', 'Flagged'],
    yticks=[0, 1],
    yticklabels=['Inactive', 'Attacking'],
    title=f'{N_RUNS} runs: all node-intervals'
)
fpr, tpr, precision, recall = curve_plot
axes[1].plot(fpr, tpr)
axes[1].set(xlabel='FPR', ylabel='TPR', title='ROC (reporting run 1)')
axes[2].plot(recall, precision)
axes[2].set(xlabel='Recall', ylabel='Precision', title='PR (reporting run 1)')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'detection_curves.png')
plt.show()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, metric in zip(axes, ['throughput', 'sla_violations', 'dropped_vms']):
    for scenario in SCENARIOS:
        threshold_statistics = threshold_sweep[threshold_sweep.scenario == scenario].groupby('threshold')[metric].agg(['mean', 'std'])
        axis.errorbar(
            threshold_statistics.index,
            threshold_statistics['mean'],
            yerr=threshold_statistics['std'],
            label=scenario,
            marker='o'
        )
    axis.set(xlabel='Threshold', ylabel=metric, title='Mean ± SD across runs')
    axis.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'threshold_sweep.png')
plt.show()
print('Last simulated ledger valid:', reference_ledger.is_valid())

In [ ]:
exports = dict(
    per_run_results=results,
    comparative_summary=summary,
    detection_per_run=detection,
    detection_summary=detection_summary,
    detection_confusion_matrix=confusion,
    honest_fpr=honest_fpr_report,
    behavior_recall=behavior_summary,
    episode_latencies=latencies,
    latency_summary=latency_summary,
    ledger_per_run=storage_results,
    ledger_summary=storage_summary,
    paired_tests=paired_tests,
    weight_ablations=ablations,
    ablation_summary=ablation_summary,
    curve_metrics=curves,
    threshold_sweep=threshold_sweep,
    storage_grid=storage_grid,
    storage_grid_summary=storage_grid_summary
)
for name, frame in exports.items():
    frame.to_csv(OUTPUT_DIR / f'{name}.csv', index=name == 'detection_confusion_matrix')
metadata = dict(
    n_runs=N_RUNS,
    n_nodes=N_NODES,
    t_intervals=T_INTERVALS,
    calibration_seeds=CALIBRATION_SEEDS,
    report_seeds=REPORT_SEEDS,
    sensitivity=SENSITIVITY,
    threshold=TRUST_THRESHOLD,
    random_weights=random_weights.tolist(),
    ahp_provenance='author-defined illustrative ratios; not expert elicitation',
    ledger='single-process simulated hash chain',
    modeled_consensus_ms=MODELED_CONSENSUS_MS,
    modeled_tps=MODELED_RECORDS_PER_SECOND,
    energy_units='model units per one-unit interval',
    memory_measure='maximum tracemalloc peak of trust computation and scheduler replay'
)
(OUTPUT_DIR / 'experiment_metadata.json').write_text(json.dumps(metadata, indent=2))
print('Exports:', OUTPUT_DIR.resolve())